In [ ]:
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from backend.model_utils import SoftVotingOnlineEnsemble
from backend.feature_engineering import build_combined_text, build_numeric_features, make_features

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "social_skills_dataset.csv"
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
required_cols = {"category", "situation", "prompt", "response_a", "response_b", "human_choice"}
missing = required_cols - set(df.columns)
assert not missing, f"Missing required columns: {missing}"

print("Null counts:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nCategory distribution:")
print(df["category"].value_counts())

print("\nHuman choice distribution:")
print(df["human_choice"].value_counts(normalize=True))

In [ ]:
before = len(df)
df = df.dropna(subset=list(required_cols)).drop_duplicates().reset_index(drop=True)
print(f"Rows before: {before} | after cleaning: {len(df)}")

In [ ]:
df["combined_text"] = build_combined_text(df["category"], df["situation"], df["prompt"], df["response_a"], df["response_b"])

numeric_features = build_numeric_features(df["response_a"], df["response_b"])

x_text = df["combined_text"]
y = df["human_choice"]

numeric_features.head()

In [ ]:
idx_trainval, idx_test = train_test_split(df.index, test_size=0.2, random_state=42, stratify=y)

idx_train, idx_val = train_test_split(idx_trainval, test_size=0.25, random_state=42, stratify=y.loc[idx_trainval])

print(f"Train: {len(idx_train)} | Validation: {len(idx_val)} | Test: {len(idx_test)}")

In [ ]:
vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)

In [ ]:
x_train_tfidf = vectorizer.fit_transform(x_text.loc[idx_train])
x_val_tfidf = vectorizer.transform(x_text.loc[idx_val])
x_test_tfidf = vectorizer.transform(x_text.loc[idx_test])

print("Vocabulary size:", len(vectorizer.get_feature_names_out()))

In [ ]:
numeric_scaler = MaxAbsScaler()
num_train = numeric_scaler.fit_transform(numeric_features.loc[idx_train])
num_val = numeric_scaler.transform(numeric_features.loc[idx_val])
num_test = numeric_scaler.transform(numeric_features.loc[idx_test])

In [ ]:
x_train_full = sp.hstack([x_train_tfidf, num_train]).tocsr()
x_val_full = sp.hstack([x_val_tfidf, num_val]).tocsr()
x_test_full = sp.hstack([x_test_tfidf, num_test]).tocsr()

y_train, y_val, y_test = y.loc[idx_train], y.loc[idx_val], y.loc[idx_test]

print("Final feature matrix shape (train):", x_train_full.shape)

In [ ]:
model = SoftVotingOnlineEnsemble([("sgd_log", SGDClassifier(loss="log_loss", random_state=42)),
                                  ("sgd_huber", SGDClassifier(loss="modified_huber", random_state=7)),
                                  ("naive_bayes", MultinomialNB()),])

model.fit(x_train_full, y_train)
print("Trained:", type(model).__name__, "with estimators:", [name for name, _ in model.estimators])

In [ ]:
val_pred = model.predict(x_val_full)
print("Validation accuracy:", f"{accuracy_score(y_val, val_pred):.2%}")

y_pred = model.predict(x_test_full)
print("Test accuracy:", f"{accuracy_score(y_test, y_pred):.2%}")
print()
print(classification_report(y_test, y_pred, target_names=["Response A", "Response B"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["A", "B"])
disp.plot()
plt.title("Social Skills — Response Preference (Test set)")
plt.show()

In [ ]:
sample = make_features(vectorizer, numeric_scaler,
                       prompt="I want to start a conversation with a new person.",
                       response_a="Ask about their day and then introduce myself.",
                       response_b="Ask how they are and then introduce myself.",)

pred = int(model.predict(sample)[0])
proba = model.predict_proba(sample)[0]
print("Prediction:", "Response A" if pred == 0 else "Response B")
print(f"P(A) = {proba[0]:.2%}  |  P(B) = {proba[1]:.2%}")

In [ ]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(model, MODELS_DIR / "social_skills_model.pkl")
joblib.dump(vectorizer, MODELS_DIR / "social_skills_vectorizer.pkl")
joblib.dump(numeric_scaler, MODELS_DIR / "social_skills_scaler.pkl")

print("Saved model, vectorizer, and scaler to", MODELS_DIR)